# Pose Estimation Evaluation — AIST++ Dataset
### Dual Benchmark: Accuracy vs Real-world Speed

**Models:**
- YOLO26s-pose (Ultralytics) — single-stage CNN
- RTMPose-m (rtmlib) — top-down CNN
- ViTPose-base (HuggingFace) — top-down Vision Transformer

**Two evaluation approaches:**

| Benchmark | Purpose | Bounding Box | Fair for |
|---|---|---|---|
| **Benchmark 1** | Accuracy (AP) | Ground truth | Pose model quality |
| **Benchmark 2** | Speed (FPS) | Own detector | Real-world deployment |

**Note on YOLO26s:** Single-stage architecture combines detection + keypoints in one forward pass. Cannot be separated, so it uses its own detector in both benchmarks. This is noted in results.

**Dataset:** AIST++ (5 sequences × 100 frames = 500 images)  
**Genres:** Break, Ballet Jazz, Street Jazz, Mid Hip-hop, LA Hip-hop  
**Metric:** COCO AP (Average Precision), AP50, AP75, AR, FPS

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA version:', torch.version.cuda)
else:
    print('Go to Runtime → Change runtime type → T4 GPU')


In [ ]:
!pip install ultralytics --quiet
print('Ultralytics (YOLO)')


In [ ]:
!pip uninstall onnxruntime onnxruntime-gpu -y --quiet
!pip install onnxruntime-gpu==1.19.0 --quiet
 
import torch
import onnxruntime as ort
providers = ort.get_available_providers()
print('Available providers:', providers)
cuda_available = 'CUDAExecutionProvider' in providers
backend_device = 'cuda' if cuda_available else 'cpu'
print(f'RTMPose will run on: {backend_device.upper()}')


In [ ]:
!pip install rtmlib --quiet
print('rtmlib (RTMPose)')


In [ ]:
!pip install transformers --quiet
!pip install pycocotools opencv-python-headless matplotlib pandas --quiet
print('All packages installed')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
 
import os, json, time
import numpy as np
import cv2
from PIL import Image
 
OUTPUT_DIR  = '/content/drive/MyDrive/AIST_Output'
FRAMES_DIR  = f'{OUTPUT_DIR}/frames'
COCO_JSON   = f'{OUTPUT_DIR}/aist_coco.json'
RESULTS_DIR = '/content/drive/MyDrive/AIST_Final_Results'
os.makedirs(RESULTS_DIR, exist_ok=True)
 
with open(COCO_JSON) as f:
    coco_data = json.load(f)
 
device = 'cuda' if torch.cuda.is_available() else 'cpu'
 
img_id_to_gt_bbox = {ann['image_id']: ann['bbox'] for ann in coco_data['annotations']}
 
print(f'Drive mounted — device: {device.upper()}')
print(f'Images: {len(coco_data["images"])}')
print(f'Annotations: {len(coco_data["annotations"])}')


In [ ]:
from ultralytics import YOLO
yolo_model = YOLO('yolo26s-pose.pt')
print('YOLO26s-pose loaded')


In [ ]:
from rtmlib import Body
rtm_model = Body(
    det='https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/yolox_m_8xb8-300e_humanart-c2c7a14a.zip',
    det_input_size=(640, 640),
    pose='https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.zip',
    pose_input_size=(192, 256),
    to_openpose=False,
    backend='onnxruntime',
    device=backend_device
)
print(f'RTMPose-m loaded on {backend_device.upper()}')


In [ ]:
from transformers import AutoModelForPoseEstimation, AutoProcessor
 
vit_processor = AutoProcessor.from_pretrained('usyd-community/vitpose-base-simple')
vit_model = AutoModelForPoseEstimation.from_pretrained('usyd-community/vitpose-base-simple')
vit_model = vit_model.to(device)
vit_model.eval()
print(f'ViTPose-base loaded on {device.upper()}')


In [ ]:
yolo_b1_predictions = []
yolo_b1_times = []
 
print(f'YOLO inference (full pipeline, {device.upper()})...')
 
for i, img_info in enumerate(coco_data['images']):
    img_path = f"{FRAMES_DIR}/{img_info['file_name']}"
    image_id = img_info['id']
 
    start = time.time()
    results = yolo_model(img_path, verbose=False)
    yolo_b1_times.append(time.time() - start)
 
    if results[0].keypoints is None or len(results[0].keypoints) == 0:
        continue
    if len(results[0].boxes) == 0:
        continue
 
    best_idx = results[0].boxes.conf.argmax().item()
    kpts      = results[0].keypoints[best_idx].xy[0].cpu().numpy()
    kpts_conf = results[0].keypoints[best_idx].conf[0].cpu().numpy()
    box       = results[0].boxes[best_idx].xyxy[0].cpu().numpy()
    score     = float(results[0].boxes[best_idx].conf[0].cpu())
 
    kpts_coco = []
    for j in range(17):
        kpts_coco.extend([float(kpts[j][0]), float(kpts[j][1]), float(kpts_conf[j])])
 
    x1, y1, x2, y2 = box
    yolo_b1_predictions.append({
        'image_id': image_id, 'category_id': 1,
        'keypoints': kpts_coco, 'score': score,
        'bbox': [float(x1), float(y1), float(x2-x1), float(y2-y1)]
    })
 
    if (i+1) % 100 == 0:
        print(f'  {i+1}/500...')
 
with open('yolo_b1.json', 'w') as f:
    json.dump(yolo_b1_predictions, f)
 
yolo_b1_fps = 1 / np.mean(yolo_b1_times)
print(f'YOLO: {len(yolo_b1_predictions)}/500 | FPS={yolo_b1_fps:.1f}')


In [ ]:
rtm_b1_predictions = []
rtm_b1_times = []
 
print(f'RTMPose inference ({backend_device.upper()})...')
 
for i, img_info in enumerate(coco_data['images']):
    img_path = f"{FRAMES_DIR}/{img_info['file_name']}"
    image_id = img_info['id']
    w, h = img_info['width'], img_info['height']
 
    img = cv2.imread(img_path)
    if img is None:
        continue
 
    start = time.time()
    keypoints, scores = rtm_model(img)
    rtm_b1_times.append(time.time() - start)
 
    if keypoints is None or len(keypoints) == 0:
        continue
 
    kpts  = keypoints[0]
    confs = scores[0]
 
    kpts_coco = []
    for j in range(17):
        kpts_coco.extend([float(kpts[j][0]), float(kpts[j][1]), float(confs[j])])
 
    rtm_b1_predictions.append({
        'image_id': image_id, 'category_id': 1,
        'keypoints': kpts_coco,
        'score': float(np.mean(confs)),
        'bbox': [0, 0, float(w), float(h)]
    })
 
    if (i+1) % 100 == 0:
        print(f'  {i+1}/500...')
 
with open('rtm_b1.json', 'w') as f:
    json.dump(rtm_b1_predictions, f)
 
rtm_b1_fps = 1 / np.mean(rtm_b1_times)
print(f'RTMPose: {len(rtm_b1_predictions)}/500 | FPS={rtm_b1_fps:.1f}')


In [ ]:
vit_b1_predictions = []
vit_b1_times = []
 
print(f'ViTPose inference with GT bbox ({device.upper()})...')
 
for i, img_info in enumerate(coco_data['images']):
    img_path = f"{FRAMES_DIR}/{img_info['file_name']}"
    image_id = img_info['id']
 
    if image_id not in img_id_to_gt_bbox:
        continue
 
    bbox = img_id_to_gt_bbox[image_id]
    boxes = [[[float(bbox[0]), float(bbox[1]), float(bbox[2]), float(bbox[3])]]]
 
    img_pil = Image.open(img_path).convert('RGB')
 
    start = time.time()
    inputs = vit_processor(img_pil, boxes=boxes, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = vit_model(**inputs)
    vit_b1_times.append(time.time() - start)
 
    results = vit_processor.post_process_pose_estimation(outputs, boxes)[0][0]
    kpts  = results['keypoints'].cpu().numpy()
    confs = results['scores'].cpu().numpy()
 
    kpts_coco = []
    for j in range(17):
        kpts_coco.extend([float(kpts[j][0]), float(kpts[j][1]), float(confs[j])])
 
    vit_b1_predictions.append({
        'image_id': image_id, 'category_id': 1,
        'keypoints': kpts_coco,
        'score': float(np.mean(confs)),
        'bbox': [float(bbox[0]), float(bbox[1]), float(bbox[2]), float(bbox[3])]
    })
 
    if (i+1) % 100 == 0:
        print(f'  {i+1}/500...')
 
with open('vit_b1.json', 'w') as f:
    json.dump(vit_b1_predictions, f)
 
vit_b1_fps = 1 / np.mean(vit_b1_times)
print(f'ViTPose: {len(vit_b1_predictions)}/500 | FPS={vit_b1_fps:.1f}')


In [ ]:
yolo_b2_predictions = yolo_b1_predictions
yolo_b2_fps = yolo_b1_fps
print(f'YOLO B2: same as B1 (single-stage, inseparable) | FPS={yolo_b2_fps:.1f}')


In [ ]:
rtm_b2_predictions = rtm_b1_predictions
rtm_b2_fps = rtm_b1_fps
print(f'RTMPose B2: same as B1 (YOLOX included) | FPS={rtm_b2_fps:.1f}')


In [ ]:
yolo_bbox_map = {p['image_id']: p['bbox'] for p in yolo_b1_predictions}
 
vit_b2_predictions = []
vit_b2_times = []
 
print(f'ViTPose B2: YOLO26s detector + ViTPose keypoints ({device.upper()})...')
 
for i, img_info in enumerate(coco_data['images']):
    img_path = f"{FRAMES_DIR}/{img_info['file_name']}"
    image_id = img_info['id']
 
    
    if image_id not in yolo_bbox_map:
        continue
 
    
    yolo_bbox = yolo_bbox_map[image_id]
    boxes = [[[float(yolo_bbox[0]), float(yolo_bbox[1]),
               float(yolo_bbox[2]), float(yolo_bbox[3])]]]
 
    img_pil = Image.open(img_path).convert('RGB')
 
    
    start = time.time()
 
    
    yolo_det = yolo_model(img_path, verbose=False)
 
    
    inputs = vit_processor(img_pil, boxes=boxes, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = vit_model(**inputs)
 
    vit_b2_times.append(time.time() - start)
 
    results = vit_processor.post_process_pose_estimation(outputs, boxes)[0][0]
    kpts  = results['keypoints'].cpu().numpy()
    confs = results['scores'].cpu().numpy()
 
    kpts_coco = []
    for j in range(17):
        kpts_coco.extend([float(kpts[j][0]), float(kpts[j][1]), float(confs[j])])
 
    vit_b2_predictions.append({
        'image_id': image_id, 'category_id': 1,
        'keypoints': kpts_coco,
        'score': float(np.mean(confs)),
        'bbox': [float(yolo_bbox[0]), float(yolo_bbox[1]),
                 float(yolo_bbox[2]), float(yolo_bbox[3])]
    })
 
    if (i+1) % 100 == 0:
        print(f'  {i+1}/500...')
 
with open('vit_b2.json', 'w') as f:
    json.dump(vit_b2_predictions, f)
 
vit_b2_fps = 1 / np.mean(vit_b2_times)
print(f'ViTPose B2: {len(vit_b2_predictions)}/500 | FPS={vit_b2_fps:.1f}')


In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
 
def evaluate(gt_json, pred_json, label):
    coco_gt = COCO(gt_json)
    coco_dt = coco_gt.loadRes(pred_json)
    ev = COCOeval(coco_gt, coco_dt, 'keypoints')
    ev.evaluate()
    ev.accumulate()
    print(f'\n=== {label} ===')
    ev.summarize()
    return {
        'AP':   round(ev.stats[0], 4),
        'AP50': round(ev.stats[1], 4),
        'AP75': round(ev.stats[2], 4),
        'AR':   round(ev.stats[5], 4),
    }
 
print('=' * 50)
print('BENCHMARK 1 — Accuracy (GT bbox)')
print('=' * 50)
yolo_b1 = evaluate(COCO_JSON, 'yolo_b1.json', 'YOLO26s-pose (full pipeline)')
rtm_b1  = evaluate(COCO_JSON, 'rtm_b1.json',  'RTMPose-m (full pipeline)')
vit_b1  = evaluate(COCO_JSON, 'vit_b1.json',  'ViTPose-base (GT bbox, pose only)')
 
print('\n' + '=' * 50)
print('BENCHMARK 2 — Real-world Speed (own detector)')
print('=' * 50)
vit_b2 = evaluate(COCO_JSON, 'vit_b2.json', 'ViTPose-base (YOLO26s + ViTPose)')


In [ ]:
genre_image_ids = {}
for img in coco_data['images']:
    genre = img.get('genre', 'Unknown')
    if genre not in genre_image_ids:
        genre_image_ids[genre] = []
    genre_image_ids[genre].append(img['id'])
 
print('Images per genre:')
for genre, ids in genre_image_ids.items():
    print(f'  {genre}: {len(ids)}')


In [ ]:
def evaluate_by_genre(gt_json_path, pred_json_path, genre_image_ids):
    with open(pred_json_path) as f:
        all_preds = json.load(f)
    with open(gt_json_path) as f:
        gt_data = json.load(f)
 
    results = {}
    for genre, img_ids in genre_image_ids.items():
        img_id_set = set(img_ids)
        genre_gt = {
            'images':      [img for img in gt_data['images'] if img['id'] in img_id_set],
            'annotations': [ann for ann in gt_data['annotations'] if ann['image_id'] in img_id_set],
            'categories':  gt_data['categories']
        }
        genre_preds = [p for p in all_preds if p['image_id'] in img_id_set]
        if not genre_preds:
            results[genre] = {'AP': 0, 'AP50': 0, 'AP75': 0}
            continue
        with open('tmp_gt.json', 'w') as f:
            json.dump(genre_gt, f)
        with open('tmp_pred.json', 'w') as f:
            json.dump(genre_preds, f)
        try:
            coco_gt   = COCO('tmp_gt.json')
            coco_dt   = coco_gt.loadRes('tmp_pred.json')
            coco_eval = COCOeval(coco_gt, coco_dt, 'keypoints')
            coco_eval.evaluate()
            coco_eval.accumulate()
            coco_eval.summarize()
            results[genre] = {
                'AP':   round(coco_eval.stats[0], 4),
                'AP50': round(coco_eval.stats[1], 4),
                'AP75': round(coco_eval.stats[2], 4),
            }
        except:
            results[genre] = {'AP': 0, 'AP50': 0, 'AP75': 0}
    return results
 
print('Evaluating by genre...')
yolo_genre = evaluate_by_genre(COCO_JSON, 'yolo_b1.json', genre_image_ids)
rtm_genre  = evaluate_by_genre(COCO_JSON, 'rtm_b1.json',  genre_image_ids)
vit_genre  = evaluate_by_genre(COCO_JSON, 'vit_b1.json',  genre_image_ids)
print('Done')


In [ ]:
import pandas as pd
 
b1_table = pd.DataFrame([
    {
        'Model': 'YOLO26s-pose', 'Architecture': 'Single-stage CNN',
        'Bbox': 'Own detector*',
        'AP': yolo_b1['AP'], 'AP50': yolo_b1['AP50'], 'AP75': yolo_b1['AP75'],
        'AR': yolo_b1['AR'], 'FPS (pose only)': round(yolo_b1_fps, 1)
    },
    {
        'Model': 'RTMPose-m', 'Architecture': 'Top-down CNN',
        'Bbox': 'Own detector',
        'AP': rtm_b1['AP'], 'AP50': rtm_b1['AP50'], 'AP75': rtm_b1['AP75'],
        'AR': rtm_b1['AR'], 'FPS (pose only)': round(rtm_b1_fps, 1)
    },
    {
        'Model': 'ViTPose-base', 'Architecture': 'Top-down Transformer',
        'Bbox': 'GT bbox',
        'AP': vit_b1['AP'], 'AP50': vit_b1['AP50'], 'AP75': vit_b1['AP75'],
        'AR': vit_b1['AR'], 'FPS (pose only)': round(vit_b1_fps, 1)
    }
])
 
print('=== TABLE 1: Benchmark 1 — Accuracy (AIST++ dataset) ===')
print('* YOLO26s cannot use external bbox (single-stage architecture)')
print(b1_table.to_string(index=False))
b1_table.to_csv('benchmark1_accuracy.csv', index=False)
 
b2_table = pd.DataFrame([
    {
        'Model': 'YOLO26s-pose', 'Pipeline': 'Detection + Pose (1 pass)',
        'AP': yolo_b1['AP'], 'FPS': round(yolo_b2_fps, 1)
    },
    {
        'Model': 'RTMPose-m', 'Pipeline': 'YOLOX + RTMPose',
        'AP': rtm_b1['AP'], 'FPS': round(rtm_b2_fps, 1)
    },
    {
        'Model': 'ViTPose-base', 'Pipeline': 'YOLO26s + ViTPose',
        'AP': vit_b2['AP'], 'FPS': round(vit_b2_fps, 1)
    }
])
 
print('\n=== TABLE 2: Benchmark 2 — Real-world Pipeline Speed ===')
print(b2_table.to_string(index=False))
b2_table.to_csv('benchmark2_speed.csv', index=False)
 
genre_rows = []
for genre in genre_image_ids.keys():
    genre_rows.append({
        'Genre': genre,
        'YOLO AP': yolo_genre[genre]['AP'],
        'YOLO AP50': yolo_genre[genre]['AP50'],
        'RTMPose AP': rtm_genre[genre]['AP'],
        'RTMPose AP50': rtm_genre[genre]['AP50'],
        'ViTPose AP': vit_genre[genre]['AP'],
        'ViTPose AP50': vit_genre[genre]['AP50'],
    })
 
genre_table = pd.DataFrame(genre_rows)
print('\n=== TABLE 3: AP by Genre ===')
print(genre_table.to_string(index=False))
genre_table.to_csv('genre_results.csv', index=False)


In [ ]:
import matplotlib.pyplot as plt
 
colors  = ['#2196F3', '#4CAF50', '#FF9800']
models  = ['YOLO26s', 'RTMPose-m', 'ViTPose-b']
genres  = list(genre_image_ids.keys())
 
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
 
metric_names = ['AP', 'AP50', 'AP75']
yv = [yolo_b1['AP'], yolo_b1['AP50'], yolo_b1['AP75']]
rv = [rtm_b1['AP'],  rtm_b1['AP50'],  rtm_b1['AP75']]
vv = [vit_b1['AP'],  vit_b1['AP50'],  vit_b1['AP75']]
 
x = range(3)
w = 0.25
axes[0,0].bar([i-w for i in x], yv, w, label='YOLO26s*',  color=colors[0], alpha=0.9)
axes[0,0].bar([i   for i in x], rv, w, label='RTMPose-m', color=colors[1], alpha=0.9)
axes[0,0].bar([i+w for i in x], vv, w, label='ViTPose-b', color=colors[2], alpha=0.9)
for i, (a,b,c) in enumerate(zip(yv, rv, vv)):
    axes[0,0].text(i-w, a+0.005, f'{a:.3f}', ha='center', fontsize=7)
    axes[0,0].text(i,   b+0.005, f'{b:.3f}', ha='center', fontsize=7)
    axes[0,0].text(i+w, c+0.005, f'{c:.3f}', ha='center', fontsize=7)
axes[0,0].set_xticks(list(x))
axes[0,0].set_xticklabels(metric_names)
axes[0,0].set_ylim(0, 1.1)
axes[0,0].set_title('Benchmark 1: Accuracy (AP)\n* YOLO uses own detector')
axes[0,0].legend()
axes[0,0].grid(axis='y', alpha=0.3)
 
fps_vals = [yolo_b2_fps, rtm_b2_fps, vit_b2_fps]
bars = axes[0,1].bar(models, fps_vals, color=colors, alpha=0.9)
for bar, v in zip(bars, fps_vals):
    axes[0,1].text(bar.get_x()+bar.get_width()/2, v+0.3,
                  f'{v:.1f}', ha='center', fontweight='bold')
axes[0,1].set_ylabel('FPS')
axes[0,1].set_title('Benchmark 2: Real-world Speed\n(each model with own detector)')
axes[0,1].grid(axis='y', alpha=0.3)
 
xg = range(len(genres))
yolo_g = [yolo_genre[g]['AP'] for g in genres]
rtm_g  = [rtm_genre[g]['AP']  for g in genres]
vit_g  = [vit_genre[g]['AP']  for g in genres]
axes[1,0].bar([i-w for i in xg], yolo_g, w, label='YOLO26s',   color=colors[0], alpha=0.9)
axes[1,0].bar([i   for i in xg], rtm_g,  w, label='RTMPose-m', color=colors[1], alpha=0.9)
axes[1,0].bar([i+w for i in xg], vit_g,  w, label='ViTPose-b', color=colors[2], alpha=0.9)
axes[1,0].set_xticks(list(xg))
axes[1,0].set_xticklabels(genres, rotation=15, ha='right', fontsize=9)
axes[1,0].set_ylim(0, 1.1)
axes[1,0].set_ylabel('AP')
axes[1,0].set_title('AP by Dance Genre')
axes[1,0].legend()
axes[1,0].grid(axis='y', alpha=0.3)
 
yolo_g50 = [yolo_genre[g]['AP50'] for g in genres]
rtm_g50  = [rtm_genre[g]['AP50']  for g in genres]
vit_g50  = [vit_genre[g]['AP50']  for g in genres]
axes[1,1].bar([i-w for i in xg], yolo_g50, w, label='YOLO26s',   color=colors[0], alpha=0.9)
axes[1,1].bar([i   for i in xg], rtm_g50,  w, label='RTMPose-m', color=colors[1], alpha=0.9)
axes[1,1].bar([i+w for i in xg], vit_g50,  w, label='ViTPose-b', color=colors[2], alpha=0.9)
axes[1,1].set_xticks(list(xg))
axes[1,1].set_xticklabels(genres, rotation=15, ha='right', fontsize=9)
axes[1,1].set_ylim(0, 1.1)
axes[1,1].set_ylabel('AP50')
axes[1,1].set_title('AP50 by Dance Genre')
axes[1,1].legend()
axes[1,1].grid(axis='y', alpha=0.3)
 
plt.suptitle('Pose Estimation Model Comparison — AIST++ Dataset (GPU T4)', fontsize=14)
plt.tight_layout()
plt.savefig('final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Charts saved')


In [ ]:
import shutil
 
files = [
    'yolo_b1.json', 'rtm_b1.json', 'vit_b1.json', 'vit_b2.json',
    'benchmark1_accuracy.csv', 'benchmark2_speed.csv',
    'genre_results.csv', 'final_comparison.png'
]
 
for f in files:
    shutil.copy(f, f'{RESULTS_DIR}/{f}')
    print(f'Saved: {f}')
 
print(f'\nAll results saved to: AIST_Final_Results/')
print()
print('=== BENCHMARK 1: Accuracy (AP) ===')
print(b1_table[['Model','AP','AP50','AP75','AR']].to_string(index=False))
print()
print('=== BENCHMARK 2: Real-world Speed ===')
print(b2_table.to_string(index=False))
